# 05 — Anomaly Resolution (M2)

Reads flagged Silver from Snowflake, applies **all Category A fixes** and **all Category B implementations**, and **overwrites corrected Silver back** to Snowflake.

**Idempotency contract:** running this notebook twice produces the same result. Corrections are guarded on the original `anomaly_reason_code` + `resolution_applied` so re-runs do not double-apply.

**Audit-trail contract (brief §8.1/§8.2):** every corrected row keeps `anomaly_flag=TRUE` and its original `anomaly_reason_code`, and gains `resolution_applied=TRUE` + `resolution_method` (+ `b_classification` for Category B). Nothing is deleted.

Order: Cat-A (A1–A16) first, then Cat-B (B1–B8), then write-back, then idempotency asserts.

In [ ]:
# --- Cell 1: install connector (Free Edition serverless; no Maven JAR) ---
%pip install -q snowflake-connector-python pandas rapidfuzz
# NOTE: dbutils.library.restartPython() intentionally omitted — it hangs on Free Edition
# serverless; %pip install works on a fresh kernel without it.

In [ ]:
# --- Cell 2: widgets + shared utils ---
import os, sys, glob
def _add_shared():
    cands = (glob.glob('/Workspace/Users/*/nexamart-m2/notebooks/_shared')
             + glob.glob('/Workspace/Users/*/nexamart_m2/notebooks/_shared')
             + [os.path.join(os.getcwd(), 'notebooks', '_shared'),
                os.path.join(os.getcwd(), '_shared')])
    for c in cands:
        if os.path.isdir(c) and c not in sys.path:
            sys.path.append(c)
            return c
    raise RuntimeError(f'_shared not found; tried: {cands}')
print('shared dir:', _add_shared())

dbutils.widgets.text('sf_account', 'rhxendw-yb24678')
dbutils.widgets.text('sf_user', 'NEXAMART_LEAD')
dbutils.widgets.text('sf_password', '')
dbutils.widgets.text('sf_warehouse', 'NEXAMART_WH')
dbutils.widgets.text('sf_role', 'NEXAMART_ENGINEER')

from pyspark.sql import functions as F
import utils_snowflake as sf
import utils_anomaly as ua   # add_resolution_columns, resolve, assert_resolved_count, RESOLUTION_METHODS
import utils_dates as ud
import utils_keys as uk

## Resolution-column contract
For each Silver table you touch: `df = ua.add_resolution_columns(df)` once after reading, then apply the
data correction (e.g. zero a revenue column) **and** `df = ua.resolve(df, cond, '<METHOD>', b_classification=...)`.
Keep `anomaly_flag` / `anomaly_reason_code` untouched.

## Category A

In [ ]:
# === A1 — CANCELLED_WITH_REVENUE (M5): zero revenue on cancelled EC orders ===
# Freeze the pre-fix condition so resolve() marks the rows BEFORE the value is zeroed.
ec = sf.read_from_snowflake(spark, 'silver_ec_orders', 'NEXAMART_SILVER')
ec = ua.add_resolution_columns(ec)
ec = ec.withColumn('_a1', (F.col('order_status') == 'CANCELLED') & (F.col('subtotal_excl_tax') > 0))
ec = ua.resolve(ec, F.col('_a1'), 'ZEROED_CANCELLED_REVENUE')
ec = ec.withColumn('subtotal_excl_tax', F.when(F.col('_a1'), F.lit(0)).otherwise(F.col('subtotal_excl_tax'))).drop('_a1')
print('A1 zeroed cancelled-with-revenue rows (expect 94):',
      ec.filter(F.col('resolution_method') == 'ZEROED_CANCELLED_REVENUE').count())

In [ ]:
# === A2 — PAYMENT_AFTER_CANCEL (M5): flag reversal-required; method FLAGGED_REVERSAL_REQUIRED ===
# TODO implement

In [ ]:
# === A3 — TAX_INCLUSION_MISMATCH (M5): normalise tax-exclusive; method NORMALISED_TAX_EXCLUSIVE ===
# TODO implement

In [ ]:
# === A4 — NL_SELLER_SOLD_AS_REVENUE (M6): relabel SELLER_SOLD events as ESTIMATED ===
# silver_nl_listing_events. These feed B6 (Estimated Classified GMV) at weight 0.60.
nle = sf.read_from_snowflake(spark, 'silver_nl_listing_events', 'NEXAMART_SILVER')
nle = ua.add_resolution_columns(nle)
a4 = F.col('event_type_code') == 'SELLER_SOLD'
nle = ua.flag(nle, a4, 'NL_SELLER_SOLD_AS_REVENUE', 'FLAGGED_ANOMALY', 'ESTIMATED')
nle = ua.resolve(nle, a4, 'RELABELLED_ESTIMATED_GMV')
print('A4 relabelled SELLER_SOLD -> ESTIMATED (expect 449):',
      nle.filter(F.col('resolution_method') == 'RELABELLED_ESTIMATED_GMV').count())

In [ ]:
# === A5 — ATP_POSITIVE_PHYSICAL_ZERO (Lead): correct ATP to 0 where physical=0 ===
# silver_wh_inventory_snapshots; M1 pre-flagged (5 rows). Flag-if-needed then fix then resolve.
wh = sf.read_from_snowflake(spark, 'silver_wh_inventory_snapshots', 'NEXAMART_SILVER')
wh = ua.add_resolution_columns(wh)
a5 = (F.col('atp_qty') > 0) & (F.col('physical_qty') == 0)
wh = ua.flag(wh, a5, 'ATP_POSITIVE_PHYSICAL_ZERO', 'FLAGGED_ANOMALY', 'UNRELIABLE')
wh = wh.withColumn('atp_qty', F.when(a5, F.lit(0)).otherwise(F.col('atp_qty')))
wh = ua.resolve(wh, a5, 'CORRECTED_ATP_TO_ZERO')
print('A5 corrected ATP->0 (expect 5):',
      wh.filter(F.col('resolution_method') == 'CORRECTED_ATP_TO_ZERO').count())

In [ ]:
# === A6 — NEGATIVE_QTY (M4): correct negative store qty to 0 + oversell flag ===
# silver_si_inventory_snapshots (A8 reconstruction writes a separate sibling table).
si = sf.read_from_snowflake(spark, 'silver_si_inventory_snapshots', 'NEXAMART_SILVER')
si = ua.add_resolution_columns(si)
a6 = (F.col('sellable_qty') < 0) | (F.col('physical_qty') < 0)
si = ua.flag(si, a6, 'NEGATIVE_QTY', 'FLAGGED_ANOMALY', 'UNRELIABLE')
si = (si
      .withColumn('oversell_flag', F.when(a6, F.lit(True)).otherwise(F.lit(False)))
      .withColumn('sellable_qty', F.when(F.col('sellable_qty') < 0, F.lit(0)).otherwise(F.col('sellable_qty')))
      .withColumn('physical_qty', F.when(F.col('physical_qty') < 0, F.lit(0)).otherwise(F.col('physical_qty'))))
si = ua.resolve(si, a6, 'CORRECTED_ATP_TO_ZERO')
print('A6 corrected negative qty->0 (expect 8):',
      si.filter(F.col('resolution_method') == 'CORRECTED_ATP_TO_ZERO').count())

In [ ]:
# === A7 — RESTOCK_BEFORE_INSPECTION (Lead): zero restock while inspection PENDING ===
# silver_rr_return_receipts. A10 (open-box) reuses THIS df (rr) — do not re-read it there.
rr = sf.read_from_snowflake(spark, 'silver_rr_return_receipts', 'NEXAMART_SILVER')
rr = ua.add_resolution_columns(rr)
a7 = (F.col('inspection_status') == 'PENDING') & (F.col('restocked_qty') > 0)
rr = ua.flag(rr, a7, 'RESTOCK_BEFORE_INSPECTION', 'FLAGGED_ANOMALY', 'UNRELIABLE')
rr = rr.withColumn('restocked_qty', F.when(a7, F.lit(0)).otherwise(F.col('restocked_qty')))
rr = ua.resolve(rr, a7, 'ZEROED_RESTOCK_PRE_INSPECTION')
print('A7 zeroed pre-inspection restock (expect 10):',
      rr.filter(F.col('resolution_method') == 'ZEROED_RESTOCK_PRE_INSPECTION').count())

In [ ]:
# === A8 — MISSING_SNAPSHOT_DAY (Lead): reconstruct stores 3/7/12 x 7 days (1-7 Aug) ===
# last-known snapshot + intervening transactions; data_quality_status='RECONSTRUCTED', certainty INFERRED;
# method RECONSTRUCTED_SNAPSHOT. (Decide: merge sibling silver_store_inventory_snapshots_reconstructed.)
# TODO implement

In [ ]:
# === A9 — SKU_PRODUCT_MISMATCH (M3): canonical product (catalogue wins); APPLIED_CANONICAL_PRODUCT ===
# TODO implement

In [ ]:
# === A10 — OPEN_BOX_AS_NEW (M4): correct open-box returns mislabelled NEW ===
# Reuses the rr dataframe from A7 (same silver_rr_return_receipts) — do NOT re-read.
a10 = (F.col('condition_on_receipt') == 'OPENED') & (F.col('restocked_as_condition') == 'NEW')
rr = ua.flag(rr, a10, 'OPEN_BOX_AS_NEW', 'FLAGGED_ANOMALY', 'UNRELIABLE')
rr = rr.withColumn('restocked_as_condition',
                   F.when(a10, F.lit('OPEN_BOX')).otherwise(F.col('restocked_as_condition')))
rr = ua.resolve(rr, a10, 'CORRECTED_CONDITION_OPEN_BOX')
print('A10 corrected open-box->NEW mislabel (expect 12):',
      rr.filter(F.col('resolution_method') == 'CORRECTED_CONDITION_OPEN_BOX').count())

In [ ]:
# === A11 — PLACEHOLDER_ID_COLLISION (M2): rekey guest customer_id 9999 to GUEST-{session} ===
# Same silver_ec_orders df as A1. M1 did not pre-flag EC orders, so establish the audit trail
# here with ua.flag(...), then rekey, then resolve. (ec already has add_resolution_columns from A1.)
ec = ec.withColumn('_a11', F.col('customer_id') == '9999')
ec = ua.flag(ec, F.col('_a11'), 'PLACEHOLDER_ID_COLLISION', 'FLAGGED_ANOMALY', 'UNRELIABLE')
ec = ec.withColumn('customer_id',
                   F.when(F.col('_a11'), F.concat(F.lit('GUEST-'), F.col('session_id')))
                    .otherwise(F.col('customer_id')))
ec = ua.resolve(ec, F.col('_a11'), 'REKEYED_GUEST_BUCKET').drop('_a11')
print('A11 rekeyed guest placeholder rows (expect 178):',
      ec.filter(F.col('resolution_method') == 'REKEYED_GUEST_BUCKET').count())

In [ ]:
# === A12 — RELISTED_AFTER_SOLD (Lead): link pair; reliability=LOW; exclude from GMV; LINKED_RELISTING_EXCLUDED ===
# TODO implement (also feeds B6 exclusion)

In [ ]:
# === A13 — IMAGE_HASH_REUSED ring (M6): multi-signal flag + risk tier; FLAGGED_FRAUD_RING ===
# TODO implement

In [ ]:
# === A14 — DELIVERY_BEFORE_SHIP (Lead) ===
# delta<=72h: corrected_ts = PICKED_UP + 36h median (CORRECTED_DELIVERY_TS)
# delta>72h : ESCALATED_MANUAL_REVIEW (do NOT auto-correct)
# TODO implement

In [ ]:
# === A15 — REVIEW_BEFORE_DELIVERY (M6): set verified_purchase=FALSE for pre-delivery reviews ===
# silver_rv_reviews IS pre-flagged by M1 (25 rows), so resolve-only (no re-flag needed).
rv = sf.read_from_snowflake(spark, 'silver_rv_reviews', 'NEXAMART_SILVER')
rv = ua.add_resolution_columns(rv)
a15 = F.col('days_post_delivery') < 0
rv = rv.withColumn('is_verified_purchase', F.when(a15, F.lit(0)).otherwise(F.col('is_verified_purchase')))
rv = ua.resolve(rv, a15, 'SET_VERIFIED_PURCHASE_FALSE')
print('A15 reviews set unverified (expect 25):',
      rv.filter(F.col('resolution_method') == 'SET_VERIFIED_PURCHASE_FALSE').count())

In [ ]:
# === A16 — DUPLICATE_CASE (M6): dedupe to canonical_case_key ===
# silver_cs_cases is pre-flagged by M1 (is_duplicate_flag, 7 rows) and already carries
# canonical_case_key. Resolve-only: stamp the audit trail; downstream complaint KPIs count
# distinct canonical_case_key so the 7 duplicates stop inflating volume.
cs = sf.read_from_snowflake(spark, 'silver_cs_cases', 'NEXAMART_SILVER')
cs = ua.add_resolution_columns(cs)
a16 = F.col('is_duplicate_flag') == True
cs = ua.resolve(cs, a16, 'DEDUPED_CANONICAL_CASE')
print('A16 duplicate cases resolved to canonical (expect 7):',
      cs.filter(F.col('resolution_method') == 'DEDUPED_CANONICAL_CASE').count())

## Category B — implement chosen interpretation + b_classification (defend in report S1)

In [ ]:
# === B1 — ATTRIBUTION (Lead): attribute 102 bridged orders; b_classification='ATTRIBUTED', conf 0.85 ===
# TODO implement

In [ ]:
# === B2 — PARTIAL_REFUND_PERIOD (Lead): recognise in return period; b_classification='RECOGNISE_IN_RETURN_PERIOD' ===
# TODO implement

In [ ]:
# === B3 — MOVEMENT_NULL_REF (Lead): b_classification='PROBABLE_MISSING_REF' (INFERRED) ===
# TODO implement

In [ ]:
# === B4 — LISTING match (M3): >=0.75 MATCHED / 0.65-0.75 MANUAL_REVIEW / <0.65 UNMATCHED (rapidfuzz) ===
# TODO implement

In [ ]:
# === B5 — IDENTITY merge (M2): probabilistic >=0.90 -> b_classification='MERGED' ===
# TODO implement

In [ ]:
# === B6 — ESTIMATED_NL_GMV (M2) ===
# weights SELLER_SOLD*0.60 + PHN_REVEAL*0.15 + CHAT*0.08 + OFFER_ACC*0.30, x listing confidence, +/-35% band.
# Exclude A12 relisted originals. Output lower/point/upper; ESTIMATED only.
# TODO implement

In [ ]:
# === B7 — BOPIS (Lead): b_classification='TREAT_AS_FULFILLED' + collection_unconfirmed flag ===
# TODO implement

In [ ]:
# === B8 — SELLER trust (M2): weighted composite -> 5 risk tiers; b_classification=tier; document weights ===
# TODO implement

## Write corrected Silver back (overwrite — idempotent)

In [ ]:
# Write corrected Silver back (overwrite — idempotent CREATE OR REPLACE via write_pandas).
# One write per corrected table. Each df accumulated ALL of that table's fixes in-memory above.
sf.write_to_snowflake(ec,  'silver_ec_orders',              'NEXAMART_SILVER', overwrite=True)  # A1, A11
sf.write_to_snowflake(wh,  'silver_wh_inventory_snapshots', 'NEXAMART_SILVER', overwrite=True)  # A5
sf.write_to_snowflake(si,  'silver_si_inventory_snapshots', 'NEXAMART_SILVER', overwrite=True)  # A6
sf.write_to_snowflake(rr,  'silver_rr_return_receipts',     'NEXAMART_SILVER', overwrite=True)  # A7, A10
sf.write_to_snowflake(nle, 'silver_nl_listing_events',      'NEXAMART_SILVER', overwrite=True)  # A4
sf.write_to_snowflake(rv,  'silver_rv_reviews',             'NEXAMART_SILVER', overwrite=True)  # A15
sf.write_to_snowflake(cs,  'silver_cs_cases',               'NEXAMART_SILVER', overwrite=True)  # A16
# TODO (author then add to write-back): pg (A2), nl (A12/A13), dc (A14), pos (A3),
#   product_master (A9), silver_store_inventory_snapshots_reconstructed (A8 sibling), and B-series.
print('Write-back complete for the resolved Silver tables.')

## Idempotency / acceptance asserts (re-run safe)

In [ ]:
# Re-read each corrected table and assert resolved counts match the targets in
# .private/resolution_targets.md. Example:
# ec2 = sf.read_from_snowflake(spark, 'silver_ec_orders', 'NEXAMART_SILVER')
# ua.assert_resolved_count(ec2, 'CANCELLED_WITH_REVENUE', 94)
# ua.assert_resolved_count(ec2, 'PLACEHOLDER_ID_COLLISION', 178)
# TODO assert all